In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="08-ad-click-prediction",
    notebook_name="04_interview_walkthrough.ipynb"
)

# 04 -- Ad Click Prediction: Complete Mock Interview Walkthrough

**Goal:** Simulate a full 45-minute ML system design interview for ad click prediction.

---

## How This Works

This notebook simulates a real interview with:
- **Interviewer** questions in blue boxes
- **Strong candidate** answers with timing guidance
- **Key phrases** highlighted that make interviewers nod approvingly
- **Common pitfalls** to avoid
- **Follow-up questions** with answers

### Timing Guide

| Step | Time | Topics |
|------|------|--------|
| 1. Clarify requirements | 3-4 min | Business goal, constraints, data |
| 2. Frame as ML task | 2-3 min | ML objective, I/O, LTR approach |
| 3. Data pipeline | 3-4 min | Data sources, label construction |
| 4. Feature engineering | 5-7 min | Ad, user, context, cross features |
| 5. Model selection | 5-7 min | LR -> DeepFM progression |
| 6. Evaluation | 3-4 min | NCE offline, CTR/revenue online |
| 7. Serving architecture | 5-7 min | 3 pipelines, continual learning |
| Follow-ups | 5-10 min | Calibration, cold start, position bias |

---

## Step 1: Clarify Requirements (3-4 minutes)

> **INTERVIEWER:** "Design an ad click prediction system for a social media platform like Facebook or Instagram."

### Strong Candidate Response:

*"Great question. Before I dive in, I'd like to clarify a few things to make sure I'm solving the right problem."*

*"First, can I assume the **business objective** is to maximize ad revenue?"*

> **INTERVIEWER:** "Yes, that's correct."

*"Second, regarding ad formats -- there are many types (image, video, carousel) and placements (timeline, stories, sidebar). For simplicity, can I focus on **image ads on the user's timeline** only?"*

> **INTERVIEWER:** "That sounds good."

*"Can the system show the **same ad to the same user** more than once? I ask because in practice, companies implement fatigue periods."*

> **INTERVIEWER:** "Yes, assume no fatigue period for simplicity."

*"For negative labels -- this is tricky in ad systems. If a user scrolls past an ad in 0.4 seconds, that shouldn't count the same as a user who stared at the ad for 10 seconds and chose not to click. Can I use **dwell time** to filter out scroll-throughs?"*

> **INTERVIEWER:** "Great question. Yes, that's a good approach."

*"Finally, **continual learning** -- in ad prediction, even a 5-minute delay in model updates can hurt performance because ad inventory and user interests change rapidly. Should I design for continual learning?"*

> **INTERVIEWER:** "Absolutely. That's a critical requirement."

### Why This Is Strong:
- Shows you understand the BUSINESS context, not just the ML
- Proactively raises the negative label challenge (most candidates miss this)
- Mentions continual learning upfront (shows production experience)

In [ ]:
# Let's organize the requirements like a staff engineer would

requirements = {
    "Business Objective": "Maximize ad revenue",
    "Ad Type": "Image ads on user timeline only",
    "Fatigue Period": "None (same ad can be shown multiple times)",
    "Negative Feedback": "Users can hide ads",
    "Negative Label Strategy": "Impression with dwell_time > t seconds but no click",
    "Continual Learning": "Critical -- 5-minute delay damages performance",
    "Scale": {
        "DAU": "~2 billion (Facebook-scale)",
        "Ads": "~10 million active ads",
        "Predictions/day": "~100 billion",
        "Latency": "< 100ms end-to-end",
    },
}

print("REQUIREMENTS SUMMARY")
print("=" * 50)
for key, value in requirements.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

---

## Step 2: Frame as ML Task (2-3 minutes)

*"Now let me frame this as an ML problem."*

*"The **ML objective** is to predict P(click | user, ad, context) -- the probability that a specific user will click a specific ad in a given context. This directly maps to the business goal because ad revenue = pCTR times bid, so better click predictions lead to higher revenue."*

*"I'd frame this as **pointwise Learning-to-Rank** with binary classification. For each (user, ad) pair, we predict a click probability, then rank ads by their expected revenue = pCTR times bid."*

```
Input:  (user_features, ad_features, context_features)
Output: P(click) in [0, 1]
Task:   Binary classification
```

### Key Phrase to Drop:
> *"We use pointwise LTR rather than pairwise because we need **well-calibrated probabilities** that feed into the ad auction, not just a correct ranking. The auction mechanism (typically second-price) requires accurate pCTR values."*

---

## Step 3: Data Pipeline (3-4 minutes)

*"We have three types of raw data:"*

1. **Ads table** -- ad ID, advertiser ID, campaign ID, category, subcategory, image/video
2. **Users table** -- user ID, demographics (age, gender, city, country), profile data
3. **Interactions table** -- user ID, ad ID, interaction type (impression, click, conversion), dwell time, timestamp

*"For constructing training labels:"*
- **Positive (y=1):** User clicks the ad within t seconds of impression
- **Negative (y=0):** Ad was visible for at least 1 second (not a scroll-through) but not clicked within t seconds
- **Discard:** Impressions with less than 1 second dwell time (user never really saw the ad)

*"This is important because naively treating all non-clicks as negatives would poison the model with examples where the user never even saw the ad."*

In [ ]:
import pandas as pd
import numpy as np

# Demonstrate label construction -- a common interview deep-dive

np.random.seed(42)

# Raw interaction logs
n_logs = 20
logs = pd.DataFrame({
    'user_id': np.random.randint(1, 6, n_logs),
    'ad_id': np.random.randint(100, 106, n_logs),
    'dwell_time_sec': np.round(np.random.exponential(3, n_logs), 1),
    'clicked': np.random.choice([0, 0, 0, 0, 1], n_logs),  # ~20% CTR for demo
})

MIN_DWELL = 1.0  # Must see ad for at least 1 second

# Apply label construction logic
logs['action'] = 'discard'  # Default
logs.loc[(logs['dwell_time_sec'] >= MIN_DWELL) & (logs['clicked'] == 1), 'action'] = 'POSITIVE'
logs.loc[(logs['dwell_time_sec'] >= MIN_DWELL) & (logs['clicked'] == 0), 'action'] = 'NEGATIVE'

print("LABEL CONSTRUCTION FROM RAW LOGS")
print("=" * 70)
print(logs.to_string(index=False))

print(f"\nSummary:")
print(f"  Positive:  {(logs['action'] == 'POSITIVE').sum()}")
print(f"  Negative:  {(logs['action'] == 'NEGATIVE').sum()}")
print(f"  Discarded: {(logs['action'] == 'discard').sum()} (dwell < {MIN_DWELL}s)")
print(f"\nDiscarded samples had dwell time < 1 second.")
print(f"The user scrolled past too fast to have actually seen the ad.")

---

## Step 4: Feature Engineering (5-7 minutes)

*"Feature engineering is where ad click prediction gets interesting. I'd organize features into four categories:"*

### 1. Ad Features
- **IDs** (advertiser, campaign, ad group, ad) -- embedded via embedding layers
- **Image/video content** -- pre-trained model (SimCLR) extracts a feature vector
- **Category and subcategory** -- from a predefined taxonomy
- **Historical stats** -- total impressions, total clicks, historical CTR

### 2. User Features
- **Demographics** -- age, gender, city, country
- **Behavioral** -- clicked ads history (embedded), total views, historical click rate
- **Contextual** -- device type, current session length, time since last click

### 3. Context Features
- Hour of day, day of week, is_weekend
- Device type, OS, connection type (WiFi vs cellular)

### 4. Cross Features (THE SECRET SAUCE)
- **user_age x ad_category** -- different age groups click different categories
- **user_country x ad_language** -- language matching
- **hour x device** -- mobile usage patterns differ by time
- **user_clicked_categories x ad_category** -- interest alignment

### Key Phrase:
> *"The feature space is extremely sparse with millions of categorical features. We use embedding layers to learn dense representations. For high-cardinality features like user_id, we apply feature hashing with a fixed bucket size."*

In [ ]:
# Visual summary of the feature pipeline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Feature Engineering Pipeline for Ad Click Prediction', fontsize=16, fontweight='bold')

# Feature source boxes
sources = [
    (0.5, 6, 'Ad Features\n\nIDs (embed)\nImage (SimCLR)\nCategory\nHistorical CTR', '#FFE0B2'),
    (3.5, 6, 'User Features\n\nDemographics\nClick history\nEngagement stats\nSession data', '#B3E5FC'),
    (6.5, 6, 'Context Features\n\nHour / day\nDevice type\nLocation\nConnection', '#C8E6C9'),
]

for x, y, text, color in sources:
    box = FancyBboxPatch((x, y), 2.5, 1.8, boxstyle='round,pad=0.1',
                        facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(x + 1.25, y + 0.9, text, ha='center', va='center', fontsize=8, fontweight='bold')

# Processing boxes
proc = [
    (0.5, 3.5, 'Embedding\nLayers\n(sparse -> dense)', '#F8BBD0'),
    (3.5, 3.5, 'Feature\nHashing\n(high cardinality)', '#D1C4E9'),
    (6.5, 3.5, 'Feature\nCrosses\n(user x ad, etc.)', '#FFF9C4'),
]

for x, y, text, color in proc:
    box = FancyBboxPatch((x, y), 2.5, 1.5, boxstyle='round,pad=0.1',
                        facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(x + 1.25, y + 0.75, text, ha='center', va='center', fontsize=8, fontweight='bold')

# Model box
model_box = FancyBboxPatch((3.5, 0.5), 5, 1.5, boxstyle='round,pad=0.1',
                           facecolor='#FFCCBC', edgecolor='#E65100', linewidth=3)
ax.add_patch(model_box)
ax.text(6, 1.25, 'DeepFM / DCN\nP(click)', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#E65100')

# Feature store
fs_box = FancyBboxPatch((9.5, 3.5), 2, 4.3, boxstyle='round,pad=0.1',
                        facecolor='#E8EAF6', edgecolor='#333', linewidth=2)
ax.add_patch(fs_box)
ax.text(10.5, 5.65, 'Feature\nStore\n\nBatch feats\n(pre-computed)\n\n+\n\nOnline feats\n(real-time)',
        ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows
for x in [1.75, 4.75, 7.75]:
    ax.annotate('', xy=(x, 5.0), xytext=(x, 6.0),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2))

for x in [1.75, 4.75, 7.75]:
    ax.annotate('', xy=(6, 2.0), xytext=(x, 3.5),
               arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

plt.tight_layout()
plt.show()

---

## Step 5: Model Selection (5-7 minutes)

*"I'd present a progression of models, starting simple:"*

1. **Logistic Regression** -- fast baseline, but cannot learn interactions
2. **Feature Crossing + LR** -- manual crosses help, but requires domain expertise and increases sparsity
3. **GBDT** -- good performance, but cannot do continual learning or train embeddings
4. **GBDT + LR** -- Facebook's classic approach. GBDT for feature extraction, then LR. Better, but still slow to update.
5. **Neural Network** -- powerful but struggles with sparse features and pairwise interactions
6. **DCN** (Google, 2017) -- cross network auto-learns feature crosses alongside a DNN
7. **FM** -- efficiently models ALL pairwise interactions via embedding dot products
8. **DeepFM** -- combines FM (pairwise) + DNN (higher-order). **This is my recommendation.**

### The Money Quote:
> *"I would start with LR as a baseline, then experiment with DCN and DeepFM. DeepFM is particularly well-suited because it efficiently captures all pairwise feature interactions through the FM component while learning complex higher-order patterns through the DNN component. Both share the same embedding layer, making it parameter-efficient. And critically, unlike GBDT, it supports continual learning via fine-tuning."*

---

## Step 6: Evaluation (3-4 minutes)

### Offline Metrics

*"For offline evaluation, I'd use two metrics:"*

1. **Cross-Entropy (CE)** -- measures how close predicted probabilities are to ground truth. Also used as the training loss.
2. **Normalized Cross-Entropy (NCE)** -- CE of our model divided by CE of a baseline that always predicts the average CTR. NCE < 1 means our model beats the baseline.

*"I specifically choose NCE over raw CE because raw CE can be misleading when CTR varies across datasets or time periods. NCE normalizes away this effect."*

### Online Metrics (A/B Testing)

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| **CTR** | clicks / impressions | Directly tied to revenue |
| **Conversion Rate** | conversions / impressions | Advertiser satisfaction |
| **Revenue Lift** | % revenue increase | Ultimate business metric |
| **Hide Rate** | hidden ads / shown ads | Measures user annoyance (false positives) |

---

## Step 7: Serving Architecture (5-7 minutes)

*"The production system has three pipelines:"*

### Pipeline 1: Data Preparation
- **Batch feature computation:** Static features (ad image embedding, category) computed periodically, stored in a feature store
- **Online feature computation:** Dynamic features (impressions in last hour, session clicks) computed at query time
- **Training data generation:** Continuously generates new labeled examples from interaction logs

### Pipeline 2: Continual Learning
- Fine-tunes the model on new training data continuously
- Evaluates the new model on a held-out validation set
- Deploys only if metrics improve (NCE, AUC)
- Critical: even a 5-minute delay can hurt performance

### Pipeline 3: Prediction
1. **Candidate Generation:** Use advertiser targeting criteria (age, gender, country) to narrow millions of ads to a small subset
2. **Ranking Service:** Fetch features (batch from store + online computed), run model to predict P(click) for each candidate
3. **Re-Ranking Service:** Apply business rules (diversity, frequency capping, ad quality)  
4. **Output:** Top-K ads, ranked by expected revenue (pCTR x bid)

### Key Design Decision:
> *"We must use online prediction -- not batch prediction -- because features like 'impressions in the last hour' and 'user's current session behavior' change in real-time. The feature store is shared between training and serving to avoid training-serving skew, which is a common source of silent model degradation."*

In [ ]:
# Visualize the three-pipeline architecture
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Ad Click Prediction: Three-Pipeline Serving Architecture', fontsize=16, fontweight='bold')

# Pipeline 1: Data Preparation
rect1 = FancyBboxPatch((0.5, 6.5), 4, 3, boxstyle='round,pad=0.1',
                        facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(rect1)
ax.text(2.5, 9.0, 'Pipeline 1: DATA PREPARATION', ha='center', fontsize=11, fontweight='bold', color='#1565C0')
ax.text(2.5, 8.2, 'Batch Features\n(ad embeddings, categories)\n\nOnline Features\n(session clicks, recency)\n\nTraining Data Gen',
        ha='center', va='center', fontsize=8)

# Pipeline 2: Continual Learning
rect2 = FancyBboxPatch((5.0, 6.5), 4, 3, boxstyle='round,pad=0.1',
                        facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(rect2)
ax.text(7, 9.0, 'Pipeline 2: CONTINUAL LEARNING', ha='center', fontsize=11, fontweight='bold', color='#2E7D32')
ax.text(7, 8.2, 'Fine-tune on new data\n(every few minutes)\n\nEvaluate on validation\n\nDeploy if improved',
        ha='center', va='center', fontsize=8)

# Feature Store
fs = FancyBboxPatch((2.5, 4.0), 3, 1.8, boxstyle='round,pad=0.1',
                     facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax.add_patch(fs)
ax.text(4, 4.9, 'FEATURE STORE\n(shared training + serving)', ha='center', fontsize=9, fontweight='bold', color='#E65100')

# Pipeline 3: Prediction
steps = [
    (0.5, 1.0, 'User\nQuery', '#FFE0B2'),
    (3.0, 1.0, 'Candidate\nGeneration\n(filter by targeting)', '#B3E5FC'),
    (6.0, 1.0, 'Ranking\nService\n(predict pCTR)', '#C8E6C9'),
    (9.0, 1.0, 'Re-Ranking\n(diversity +\nbusiness rules)', '#F8BBD0'),
    (12.0, 1.0, 'Top K\nAds', '#FFE0B2'),
]

for x, y, text, color in steps:
    box = FancyBboxPatch((x, y), 2.2, 2.0, boxstyle='round,pad=0.1',
                        facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(x + 1.1, y + 1.0, text, ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows for prediction pipeline
for x1, x2 in [(2.7, 3.0), (5.2, 6.0), (8.2, 9.0), (11.2, 12.0)]:
    ax.annotate('', xy=(x2, 2.0), xytext=(x1, 2.0),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Connection arrows
ax.annotate('', xy=(4, 5.8), xytext=(4, 6.5),
           arrowprops=dict(arrowstyle='<->', color='#E65100', lw=1.5, ls='--'))
ax.annotate('', xy=(7.1, 3.0), xytext=(7.1, 6.5),
           arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5, ls='--'))
ax.annotate('', xy=(5.0, 4.5), xytext=(7.1, 2.5),
           arrowprops=dict(arrowstyle='<-', color='#E65100', lw=1.5, ls='--'))

ax.text(7, 0.3, 'PREDICTION PIPELINE (real-time, <100ms)', ha='center', fontsize=12,
        fontweight='bold', color='#333',
        bbox=dict(boxstyle='round', facecolor='#EEEEEE', alpha=0.8))

plt.tight_layout()
plt.show()

---

## Common Follow-Up Questions

### Q1: "How do you handle position bias?"

> *"Position bias means ads in higher positions get more clicks regardless of quality. I'd address this in two ways:*
> 1. *Include position as a feature during training, then set it to a fixed value during inference so the model ranks ads by inherent quality.*
> 2. *Use inverse propensity weighting (IPW) -- weight each training example by 1/P(click|position) to debias the labels."*

### Q2: "How do you ensure model calibration?"

> *"Calibration is critical because pCTR feeds directly into the auction. I'd use:*
> - *Platt scaling (fit a sigmoid on validation predictions)*
> - *Monitor NCE as an offline metric (NCE < 1 means well-calibrated)*
> - *After negative downsampling, apply the analytical correction formula: p_true = p_sampled / (p_sampled + (1-p_sampled)/sampling_rate)"*

### Q3: "How do you handle cold-start ads?"

> *"New ads have zero interaction data. I'd use:*
> 1. *Content-based features (ad image embedding, category, text) that work without historical data*
> 2. *Advertiser-level features (historical CTR of the advertiser's other ads)*
> 3. *Exploration strategy: show new ads to a small percentage of traffic to collect initial data (epsilon-greedy or Thompson sampling)"*

### Q4: "Why not just use a deep neural network?"

> *"Pure DNNs struggle with two aspects of ad click prediction:*
> 1. *Sparse features: with millions of one-hot features, most input is zeros, making it hard for DNNs to learn*
> 2. *Pairwise interactions: DNNs can learn interactions, but not as explicitly or efficiently as FM. That is why DeepFM combines both -- FM handles the pairwise interactions, DNN handles higher-order patterns."*

### Q5: "What about data leakage?"

> *"In ranking systems, temporal data leakage is a real risk. I'd ensure:*
> - *Strict temporal train/test split -- never train on future data*
> - *Features like 'ad historical CTR' must use only data from before the timestamp of the training example*
> - *Monitor for feature-label leakage: if a feature has suspiciously high importance, investigate if it contains label information"*

In [ ]:
# === COMPLETE INTERVIEW CHEAT SHEET ===

cheat_sheet = """
====================================================================
          AD CLICK PREDICTION -- INTERVIEW CHEAT SHEET
====================================================================

30-SECOND PITCH:
"Ad click prediction estimates P(click | user, ad, context) to 
maximize revenue. I'd frame it as pointwise LTR with binary 
classification. Key features are ad IDs/categories (embedded), 
user demographics/history, and contextual signals. I'd start with 
LR baseline, then DeepFM which combines FM for pairwise interactions
with DNN for higher-order features. For evaluation: NCE offline, 
CTR/revenue lift online. Serving uses 3 pipelines: data prep, 
continual learning, and prediction (candidate gen -> rank -> re-rank)."

KEY NUMBERS:
  - Typical CTR: 1-5%
  - Latency budget: <100ms
  - Feature dimensions: millions (sparse)
  - Model update delay tolerance: <5 minutes
  - Revenue impact of 0.1% AUC improvement: $millions

VOCABULARY TO USE:
  - "Pointwise Learning-to-Rank"
  - "Sparse feature space" / "Embedding layers"
  - "Feature crosses" / "Pairwise interactions"
  - "Continual learning" / "Online fine-tuning"
  - "Calibration" (Platt scaling, NCE)
  - "Training-serving skew" / "Feature store"
  - "Negative downsampling" + "calibration correction"
  - "Position bias" / "Inverse propensity weighting"

PITFALLS TO AVOID:
  x "I'd use a deep neural network" (too vague, misses sparsity)
  x Forgetting continual learning (CRITICAL for ad systems)
  x Not mentioning calibration (breaks the auction)
  x One-hot encoding user IDs (memory explosion)
  x Using only offline metrics (must mention CTR, revenue lift)
  x Batch prediction (need real-time for dynamic features)
====================================================================
"""

print(cheat_sheet)

## Key Takeaways

1. **Always start with requirements clarification** -- business goal, ad type, negative labels, continual learning
2. **Frame clearly** -- pointwise LTR, binary classification, calibrated probabilities for auction
3. **Feature engineering is king** -- sparse features, embeddings, crosses, batch vs real-time split
4. **Model selection story** -- LR baseline -> DeepFM/DCN (explain WHY each step)
5. **NCE for offline, CTR/revenue for online** -- always mention both
6. **Three-pipeline architecture** -- data prep, continual learning, prediction
7. **Continual learning is non-negotiable** -- 5-minute delay hurts performance
8. **Calibration matters** -- pCTR feeds into the auction, must be accurate not just well-ranked

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)